# COVID-19 Data Exploration (Early Years)

This notebook explores COVID-19 death and infection data across countries during the pandemic's early years.

**Techniques Used:** Aggregate Functions, Window Functions, CTEs, Temp Tables, Joins, Views, Data Type Conversions


---

In [0]:
SELECT *
FROM users.w196717.covid_deaths
WHERE continent IS NOT NULL
ORDER BY 3, 4

## Key Metrics Overview
Selecting the core columns we'll use throughout this analysis.

In [0]:
SELECT
  location,
  date,
  total_cases,
  new_cases,
  total_deaths,
  population
FROM users.w196717.covid_deaths
WHERE continent IS NOT NULL
ORDER BY 1, 2

## Death Rate Analysis (United States)
Calculating the likelihood of dying after contracting COVID-19 — expressed as `(total_deaths / total_cases) * 100`.

In [0]:
SELECT
  location,
  date,
  total_cases,
  total_deaths,
  ROUND((total_deaths / total_cases) * 100, 2) AS death_percentage
FROM users.w196717.covid_deaths
WHERE location ILIKE '%states%'
  AND continent IS NOT NULL
ORDER BY 1, 2

## Infection Rate vs Population (United States)
What percentage of the U.S. population has been infected over time?

In [0]:
SELECT
  location,
  date,
  population,
  total_cases,
  ROUND((total_cases / population) * 100, 4) AS pct_population_infected
FROM users.w196717.covid_deaths
WHERE location ILIKE '%states%'
ORDER BY 1, 2

## Global Comparison: Highest Infection Rates
Ranking all countries by their peak infection rate relative to population size.

In [0]:
SELECT
  location,
  population,
  MAX(total_cases)                        AS highest_infection_count,
  ROUND(MAX(total_cases / population) * 100, 2) AS pct_population_infected
FROM users.w196717.covid_deaths
WHERE continent IS NOT NULL
GROUP BY location, population
ORDER BY pct_population_infected DESC

## Death Count Rankings
Identifying countries and continents with the highest total death counts.

In [0]:
SELECT
  location,
  MAX(CAST(total_deaths AS INT)) AS total_death_count
FROM users.w196717.covid_deaths
WHERE continent IS NOT NULL
GROUP BY location
ORDER BY total_death_count DESC

In [0]:
SELECT
  continent,
  MAX(CAST(total_deaths AS INT)) AS total_death_count
FROM users.w196717.covid_deaths
WHERE continent IS NOT NULL
GROUP BY continent
ORDER BY total_death_count DESC

## Global Totals
Aggregate worldwide case and death figures with overall death percentage.

In [0]:
SELECT
  SUM(new_cases)                                          AS total_cases,
  SUM(CAST(new_deaths AS INT))                            AS total_deaths,
  ROUND(SUM(CAST(new_deaths AS INT)) / SUM(new_cases) * 100, 2) AS death_percentage
FROM users.w196717.covid_deaths
WHERE continent IS NOT NULL
ORDER BY 1, 2

## Vaccination Progress
Joining deaths and vaccinations data to track rolling vaccination counts and percentage of population vaccinated.

---

In [0]:
SELECT
  dea.continent,
  dea.location,
  dea.date,
  dea.population,
  vac.new_vaccinations,
  SUM(CAST(vac.new_vaccinations AS INT)) OVER (
    PARTITION BY dea.location
    ORDER BY dea.date
  ) AS rolling_people_vaccinated
FROM users.w196717.covid_deaths dea
JOIN users.w196717.covid_vaccinations vac
  ON dea.location = vac.location
  AND dea.date = vac.date
WHERE dea.continent IS NOT NULL
ORDER BY 2, 3

In [0]:
WITH PopvsVac AS (
  SELECT
    dea.continent,
    dea.location,
    dea.date,
    dea.population,
    vac.new_vaccinations,
    SUM(CAST(vac.new_vaccinations AS INT)) OVER (
      PARTITION BY dea.location
      ORDER BY dea.date
    ) AS rolling_people_vaccinated
  FROM users.w196717.covid_deaths dea
  JOIN users.w196717.covid_vaccinations vac
    ON dea.location = vac.location
    AND dea.date = vac.date
  WHERE dea.continent IS NOT NULL
)
SELECT
  *,
  ROUND((rolling_people_vaccinated / population) * 100, 2) AS pct_vaccinated
FROM PopvsVac

In [0]:
CREATE OR REPLACE TEMP VIEW PercentPopulationVaccinated AS
SELECT
  dea.continent,
  dea.location,
  dea.date,
  dea.population,
  vac.new_vaccinations,
  SUM(CAST(vac.new_vaccinations AS INT)) OVER (
    PARTITION BY dea.location
    ORDER BY dea.date
  ) AS rolling_people_vaccinated
FROM users.w196717.covid_deaths dea
JOIN users.w196717.covid_vaccinations vac
  ON dea.location = vac.location
  AND dea.date = vac.date;

SELECT
  *,
  ROUND((rolling_people_vaccinated / population) * 100, 2) AS pct_vaccinated
FROM PercentPopulationVaccinated